# Aula 21 · Lab bônus: orquestração modular (resolução comentada)

## 1 e 2. Os jobs "módulo"
Crie cada um como no lab da aula 12, só com as tarefas da sua etapa:

| Job | Tarefas | Parâmetro |
|---|---|---|
| `growdev_ingestao` | `preparar_ambiente` → `bronze` | `catalogo` = `growdev` |
| `growdev_transformacao` | `silver` → `gold` | `catalogo` = `growdev` |

## 3. O orquestrador
1. **Create → Job** → `growdev_orquestrador` → **Job parameters**: `catalogo` = `growdev`.
2. Tarefa `executar_ingestao`:
   - **Type:** Run Job · **Job:** `growdev_ingestao`
   - **Job parameters:** `catalogo` = `{{job.parameters.catalogo}}`
3. Tarefa `executar_transformacao`:
   - **Type:** Run Job · **Job:** `growdev_transformacao` · **Depends on:** `executar_ingestao`
   - **Job parameters:** `catalogo` = `{{job.parameters.catalogo}}`

> `{{job.parameters.catalogo}}` repassa o valor do orquestrador. Rode o orquestrador com outro catálogo e os dois módulos acompanham.

## 4. Executando
**Run now** no orquestrador. Na execução, clique em `executar_ingestao` → **link para a execução do job filho**. Cada módulo tem o seu próprio histórico, e o orquestrador mostra o fluxo completo.

## Como código
O mesmo resultado está em `resources/growdev_orquestrador.job.yml` (lá o orquestrador já inclui a etapa extra da aula 22):

```yaml
    growdev_orquestrador:
      name: growdev_orquestrador
      parameters:
        - name: catalogo
          default: growdev
      tasks:
        - task_key: executar_ingestao
          run_job_task:
            job_id: ${resources.jobs.growdev_ingestao.id}
            job_parameters:
              catalogo: "{{job.parameters.catalogo}}"
        - task_key: executar_transformacao
          depends_on:
            - task_key: executar_ingestao
          run_job_task:
            job_id: ${resources.jobs.growdev_transformacao.id}
            job_parameters:
              catalogo: "{{job.parameters.catalogo}}"
```